# Day 1: Catastrophic Forgetting Pipeline
**Split CIFAR-10** — Task A = classes 0–4, Task B = classes 5–9

This notebook trains ResNet-18 on Task A, then fine-tunes on Task B while tracking Task A accuracy decay. Checkpoints are saved at regular intervals for Day 2 layer-wise probe and CKA analysis.

## 1. Imports & Config

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader, Subset

DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR       = "./data"
CHECKPOINT_DIR = "./checkpoints"
RESULTS_DIR    = "./results"

TASK_A_CLASSES = [0, 1, 2, 3, 4]   # airplane, automobile, bird, cat, deer
TASK_B_CLASSES = [5, 6, 7, 8, 9]   # dog, frog, horse, ship, truck

TASK_A_EPOCHS  = 20
TASK_B_EPOCHS  = 20
BATCH_SIZE     = 128
LR             = 0.01
MOMENTUM       = 0.9
WEIGHT_DECAY   = 5e-4
CHECKPOINT_EVERY = 2   # save a checkpoint every N epochs during Task B

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Using device: {DEVICE}")

## 2. Data Loading

CIFAR-10 is split into two disjoint 5-class tasks. Labels are remapped to 0–4 within each task so the model always outputs 5 logits.

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])

full_train = torchvision.datasets.CIFAR10(root=DATA_DIR, train=True,
                                          download=True, transform=transform_train)
full_test  = torchvision.datasets.CIFAR10(root=DATA_DIR, train=False,
                                          download=True, transform=transform_test)


def make_subset(dataset, classes):
    """Subset containing only the given classes, labels remapped to 0..len(classes)-1."""
    indices   = [i for i, (_, label) in enumerate(dataset) if label in classes]
    label_map = {orig: new for new, orig in enumerate(classes)}

    class RemappedSubset(torch.utils.data.Dataset):
        def __init__(self, subset, label_map):
            self.subset    = Subset(dataset, indices)
            self.label_map = label_map
        def __len__(self):
            return len(self.subset)
        def __getitem__(self, idx):
            x, y = self.subset[idx]
            return x, self.label_map[y]

    return RemappedSubset(None, label_map)


task_a_train = make_subset(full_train, TASK_A_CLASSES)
task_a_test  = make_subset(full_test,  TASK_A_CLASSES)
task_b_train = make_subset(full_train, TASK_B_CLASSES)
task_b_test  = make_subset(full_test,  TASK_B_CLASSES)

loader_a_train = DataLoader(task_a_train, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
loader_a_test  = DataLoader(task_a_test,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
loader_b_train = DataLoader(task_b_train, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
loader_b_test  = DataLoader(task_b_test,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Task A train: {len(task_a_train)} | Task A test: {len(task_a_test)}")
print(f"Task B train: {len(task_b_train)} | Task B test: {len(task_b_test)}")

## 3. Model & Training Helpers

In [ ]:
def build_model(num_classes=5):
    """ResNet-18 with random init; final FC replaced for num_classes."""
    model    = torchvision.models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(DEVICE)


def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out  = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct    += (out.argmax(1) == y).sum().item()
        total      += x.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            correct += (model(x).argmax(1) == y).sum().item()
            total   += x.size(0)
    return correct / total


def evaluate_task_a_with_frozen_head(model, head, loader):
    """
    Evaluate Task A accuracy using the original Task A FC head on top of
    the current (drifting) feature extractor. This isolates representational
    forgetting from classifier forgetting.
    """
    model.eval()
    head.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y    = x.to(DEVICE), y.to(DEVICE)
            features = []
            hook     = model.avgpool.register_forward_hook(
                           lambda m, i, o: features.append(o))
            _        = model(x)
            hook.remove()
            feat     = torch.flatten(features[0], 1)
            correct += (head(feat).argmax(1) == y).sum().item()
            total   += x.size(0)
    return correct / total

## 4. Phase 1 — Train on Task A

Train ResNet-18 from random init on Task A (classes 0–4). The final checkpoint is saved as the baseline for CKA comparison on Day 2.

In [ ]:
model     = build_model(num_classes=5)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LR,
                      momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TASK_A_EPOCHS)

task_a_pretrain_acc = []

for epoch in range(1, TASK_A_EPOCHS + 1):
    loss, train_acc = train_epoch(model, loader_a_train, optimizer, criterion)
    test_acc        = evaluate(model, loader_a_test)
    task_a_pretrain_acc.append(test_acc)
    scheduler.step()
    print(f"Epoch {epoch:2d}/{TASK_A_EPOCHS} | loss {loss:.4f} | "
          f"train {train_acc:.3f} | test {test_acc:.3f}")

torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, "task_a_final.pt"))
print(f"\nTask A final test accuracy: {task_a_pretrain_acc[-1]:.3f}")
print("Saved: checkpoints/task_a_final.pt")

## 5. Phase 2 — Fine-tune on Task B (Induce Forgetting)

The FC head is replaced for 5 new classes. Task A accuracy is tracked each epoch using the **frozen original Task A head** on top of the drifting feature extractor — this isolates representational forgetting from classifier forgetting.

In [ ]:
# Replace FC head for Task B; backbone carries Task A weights
model.fc  = nn.Linear(model.fc.in_features, 5).to(DEVICE)
optimizer = optim.SGD(model.parameters(), lr=LR,
                      momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TASK_B_EPOCHS)

# Frozen Task A head for representational forgetting measurement
task_a_weights = torch.load(os.path.join(CHECKPOINT_DIR, "task_a_final.pt"),
                            map_location=DEVICE)
task_a_head        = nn.Linear(512, 5).to(DEVICE)
task_a_head.weight = nn.Parameter(task_a_weights["fc.weight"].clone())
task_a_head.bias   = nn.Parameter(task_a_weights["fc.bias"].clone())

task_a_decay_acc  = [task_a_pretrain_acc[-1]]   # epoch 0 = post-Task-A baseline
task_b_train_acc  = []
checkpoints_saved = []

for epoch in range(1, TASK_B_EPOCHS + 1):
    loss, b_train = train_epoch(model, loader_b_train, optimizer, criterion)
    b_test        = evaluate(model, loader_b_test)
    a_test        = evaluate_task_a_with_frozen_head(model, task_a_head, loader_a_test)

    task_b_train_acc.append(b_train)
    task_a_decay_acc.append(a_test)
    scheduler.step()

    print(f"Epoch {epoch:2d}/{TASK_B_EPOCHS} | "
          f"Task B train {b_train:.3f} | Task B test {b_test:.3f} | "
          f"Task A test {a_test:.3f}")

    if epoch % CHECKPOINT_EVERY == 0:
        path = os.path.join(CHECKPOINT_DIR, f"taskB_epoch{epoch:02d}.pt")
        torch.save(model.state_dict(), path)
        checkpoints_saved.append((epoch, path))
        print(f"  → Saved: {path}")

torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, "post_forgetting.pt"))
print(f"\nTask A accuracy after forgetting: {task_a_decay_acc[-1]:.3f}")
print("Saved: checkpoints/post_forgetting.pt")

## 6. Plot: Accuracy Decay Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: Task A decay
ax = axes[0]
ax.plot(range(TASK_B_EPOCHS + 1), task_a_decay_acc,
        color="#d62728", linewidth=2, marker="o", markersize=4)
ax.axhline(task_a_pretrain_acc[-1], color="#d62728",
           linewidth=1, linestyle="--", alpha=0.5, label="Task A baseline")
ax.set_xlabel("Task B Epoch")
ax.set_ylabel("Task A Test Accuracy")
ax.set_title("Task A Accuracy Decay During Task B Training")
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(True, alpha=0.3)

# Right: Task B learning
ax = axes[1]
ax.plot(range(1, TASK_B_EPOCHS + 1), task_b_train_acc,
        color="#1f77b4", linewidth=2, marker="o", markersize=4)
ax.set_xlabel("Task B Epoch")
ax.set_ylabel("Task B Train Accuracy")
ax.set_title("Task B Learning Curve")
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "day1_decay_curve.png"), dpi=150)
plt.show()
print("Saved: results/day1_decay_curve.png")

## 7. Save Arrays for Day 2

In [ ]:
np.save(os.path.join(RESULTS_DIR, "task_a_decay_acc.npy"),   np.array(task_a_decay_acc))
np.save(os.path.join(RESULTS_DIR, "task_b_train_acc.npy"),   np.array(task_b_train_acc))
np.save(os.path.join(RESULTS_DIR, "task_a_pretrain_acc.npy"), np.array(task_a_pretrain_acc))

print("Saved arrays to results/")
print("\nDay 1 complete. Checkpoints available for Day 2:")
for epoch, path in checkpoints_saved:
    print(f"  Task B epoch {epoch:2d} -> {path}")
print("  checkpoints/task_a_final.pt")
print("  checkpoints/post_forgetting.pt")